<a href="https://colab.research.google.com/github/fatwahazjuang/latihan-branch/blob/main/capstone_webscraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Web Scraping **fatsecret.co.id**

In [11]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time


def scrape_nutrition_data(query_list):
    all_data = []
    base_url = "https://www.fatsecret.co.id"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }

    for item in query_list:
        search_url = f"{base_url}/kalori-gizi/search?q={item}"

        row = {
            'nama makanan': item,
            'kalori': '-',
            'lemak': '-',
            'karbohidrat': '-',
            'protein': '-',
            'satuan': '-',
            'link': '-',
            'status': '0'
        }

        try:
            # ======================
            # 1️⃣ Pencarian
            # ======================
            res = requests.get(search_url, headers=headers)
            soup = BeautifulSoup(res.text, 'html.parser')

            # 🔹 Jika tidak ada hasil pencarian
            if soup.find('div', class_='searchNoResult'):
                print(f"❌ Tidak ditemukan: {item}")
                all_data.append(row)
                continue

            table = soup.find('table', class_='generic searchResult')
            if not table:
                print(f"❌ Tidak ditemukan (table kosong): {item}")
                all_data.append(row)
                continue

            first_link = table.find('a', class_='prominent')
            if not first_link:
                print(f"❌ Tidak ditemukan (link kosong): {item}")
                all_data.append(row)
                continue

            detail_url = base_url + first_link['href']

            # ======================
            # 2️⃣ Masuk halaman detail
            # ======================
            res = requests.get(detail_url, headers=headers)
            soup = BeautifulSoup(res.text, 'html.parser')

            # ======================
            # 3️⃣ Cek link 100 gram
            # ======================
            link_100g = soup.find('a', string=lambda s: s and "100 gram" in s.lower())

            if link_100g and link_100g.get('href'):
                detail_url = base_url + link_100g['href']
                res = requests.get(detail_url, headers=headers)
                soup = BeautifulSoup(res.text, 'html.parser')

            row['link'] = detail_url

            # ======================
            # 4️⃣ Ambil satuan dari div
            # ======================
            serving_div = soup.select_one('div.serving_size.black.serving_size_value')

            if serving_div:
                row['satuan'] = serving_div.text.strip()
            else:
                row['satuan'] = '-'

            # ======================
            # 5️⃣ Ambil nilai gizi
            # ======================
            facts = soup.find_all('td', class_='fact')

            for fact in facts:
                title_el = fact.find('div', class_='factTitle')
                value_el = fact.find('div', class_='factValue')

                if title_el and value_el:
                    title = title_el.text.strip().lower()
                    value = value_el.text.strip()

                    if 'kal' in title:
                        row['kalori'] = value
                    elif 'lemak' in title:
                        row['lemak'] = value
                    elif 'karb' in title:
                        row['karbohidrat'] = value
                    elif 'prot' in title:
                        row['protein'] = value

            if facts:
                row['status'] = '1'
            else:
                print(f"⚠️ Data gizi kosong: {item}")

        except Exception as e:
            print(f"Error pada {item}: {e}")
            row['status'] = 'Error'

        all_data.append(row)
        time.sleep(1.5)

    df = pd.DataFrame(all_data)
    df.to_csv('hasil_gizi_100gram.csv', index=False, encoding='utf-8-sig')

    print("\nSelesai! Data berhasil disimpan dengan kolom satuan.")


def preprocess_query(query_list):
    cleaned_list = []
    for item in query_list:
        item = item.replace("_", " ")
        item = item.strip()
        cleaned_list.append(item)
    return cleaned_list


# ======================
# 🔹 RUN PROGRAM
# ======================

daftar_cari = [
    'AW cola', 'Beijing Beef', 'Fried Rice', 'Hashbrown',
    'Honey Walnut Shrimp', 'Kung Pao Chicken', 'String Bean Chicken Breast',
    'Super Greens', 'The Original Orange Chicken', 'White Steamed Rice',
    'black pepper rice bowl', 'burger', 'carrot_eggs', 'cheese burger',
    'chicken waffle', 'chicken_nuggets', 'chinese_cabbage', 'chinese_sausage',
    'crispy corn', 'curry', 'french fries', 'fried chicken', 'fried_chicken',
    'fried_dumplings', 'fried_eggs', 'mango chicken pocket', 'mozza burger',
    'mung_bean_sprouts', 'nugget', 'perkedel', 'rice', 'sprite',
    'tostitos cheese dip sauce', 'triangle_hash_brown', 'water_spinach', 'shgjdshghsj', 'fgreg'
]

daftar_cari = preprocess_query(daftar_cari)

scrape_nutrition_data(daftar_cari)

❌ Tidak ditemukan: shgjdshghsj
❌ Tidak ditemukan: fgreg

Selesai! Data berhasil disimpan dengan kolom satuan.
